In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["UNSTRUCTURED_DISABLE_NOTICE"] = "1"
import logging
logging.getLogger("unstructured").setLevel(logging.ERROR)
import langchain
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document
# from langchain.chains import RetrievalQA
# New LCEL
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import DirectoryLoader
from langchain_openai import ChatOpenAI
langchain.__version__

'1.5.0'

In [2]:
with open("data/secrets/key.secret") as f:
    os.environ["OPENAI_API_KEY"] = f.read()
langchain.verbose = False

In [3]:
loader = DirectoryLoader("data/books/")
# Recommencer avec data/wiki
docs = loader.load()
len(docs)

5

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
len(all_splits)

467

In [5]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = InMemoryVectorStore(embeddings)
db = vector_store.add_documents(documents=all_splits)
len(db)

467

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
# qa_chain

In [12]:
query = "A quel age est mort Léonard de Vinci ?"
# response = qa_chain.invoke(query)
# response

In [13]:
prompt = ChatPromptTemplate.from_template(
    "Réponds à la question en te basant uniquement sur le contexte suivant :\n{context}\n\nQuestion : {question}"
)

In [14]:
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
response = rag_chain.invoke(query)
response

"Léonard de Vinci est mort à l'âge de soixante-sept ans."